In [1]:
!pip install deepchem catboost openpyxl -q


[notice] A new release of pip is available: 23.2.1 -> 24.0
[notice] To update, run: python -m pip install --upgrade pip


In [47]:
import warnings

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
import json
import random
import sys

import deepchem as dc
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import RandomizedSearchCV

In [3]:
from typing import List
from pathlib import Path
import os
from bionemo.utils.hydra import load_model_config
from bionemo.triton.utils import load_model_for_inference

BIONEMO_HOME: Path = Path(os.environ['BIONEMO_HOME']).absolute()
config_path = BIONEMO_HOME / "examples" / "molecule" / "megamolbart" / "conf"
print(f"Using model configuration at: {config_path}")

INFO:datasets:PyTorch version 2.1.0a0+32f93b1 available.


[NeMo I 2024-06-19 18:44:16 megatron_hiddens:110] Registered hidden transform sampled_var_cond_gaussian at bionemo.model.core.hiddens_support.SampledVarGaussianHiddenTransform
[NeMo I 2024-06-19 18:44:16 megatron_hiddens:110] Registered hidden transform interp_var_cond_gaussian at bionemo.model.core.hiddens_support.InterpVarGaussianHiddenTransform
Using model configuration at: /workspace/bionemo/examples/molecule/megamolbart/conf


In [6]:
cfg = load_model_config(config_name="infer.yaml", config_path=config_path)

In [9]:
cfg['model']['downstream_task']['restore_from_path'] = '/workspace/bionemo/MegaMolBART_0_2_0.nemo'

In [10]:
inferer = load_model_for_inference(cfg, interactive=True)

[NeMo I 2024-06-19 18:45:10 utils:487] pytorch DDP is not initialized. Initializing with pytorch-lightening...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


[NeMo I 2024-06-19 18:45:11 utils:333] Restoring model from /workspace/bionemo/MegaMolBART_0_2_0.nemo
[NeMo I 2024-06-19 18:45:11 utils:337] Loading model class: bionemo.model.molecule.megamolbart.megamolbart_model.MegaMolBARTModel


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Interactive mode selected, using strategy='auto'
[NeMo I 2024-06-19 18:45:11 exp_manager:394] Experiments will be logged at /workspace/bionemo/examples/molecule/megamolbart/nbs/nemo_experiments/MegaMolBART_Inference/2024-06-19_18-45-11
[NeMo I 2024-06-19 18:45:11 exp_manager:835] TensorboardLogger has been set up
[NeMo I 2024-06-19 18:45:11 utils:306] 
    
    ************** Trainer configuration ***********
[NeMo I 2024-06-19 18:45:11 utils:307] 
    name: MegaMolBART_Inference
    desc: Minimum configuration for initializing a MegaMolBART model for inference.
    trainer:
      precision: 16-mixed
      devices: 1
      num_nodes: 1
      accelerator: gpu
      logger: false
      accumulate_grad_batches: 1
    exp_manager:
      explicit_log_dir: null
      exp_dir: null
      name: ${name}
      create_checkpoint_callback: false
    model:
      name: small_span_aug
      global_batch_size: 128
      micro_batch_size: ${model.data.batch_size}
      tensor_model_parallel_size: 1
  

...

[NeMo I 2024-06-19 18:45:11 megatron_init:234] Rank 0 has data parallel group: [0]
[NeMo I 2024-06-19 18:45:11 megatron_init:237] All data parallel group ranks: [[0]]
[NeMo I 2024-06-19 18:45:11 megatron_init:238] Ranks 0 has data parallel rank: 0
[NeMo I 2024-06-19 18:45:11 megatron_init:246] Rank 0 has model parallel group: [0]
[NeMo I 2024-06-19 18:45:11 megatron_init:247] All model parallel group ranks: [[0]]
[NeMo I 2024-06-19 18:45:11 megatron_init:257] Rank 0 has tensor model parallel group: [0]
[NeMo I 2024-06-19 18:45:11 megatron_init:261] All tensor model parallel group ranks: [[0]]
[NeMo I 2024-06-19 18:45:11 megatron_init:262] Rank 0 has tensor model parallel rank: 0
[NeMo I 2024-06-19 18:45:11 megatron_init:276] Rank 0 has pipeline model parallel group: [0]
[NeMo I 2024-06-19 18:45:11 megatron_init:288] Rank 0 has embedding group: [0]
[NeMo I 2024-06-19 18:45:11 megatron_init:294] All pipeline model parallel group ranks: [[0]]
[NeMo I 2024-06-19 18:45:11 megatron_init:295]

[NeMo W 2024-06-19 18:45:11 megatron_base_model:821] The model: MegaMolBARTModel() does not have field.name: context_parallel_size in its cfg. Add this key to cfg or config_mapping to make to make it configurable.
[NeMo W 2024-06-19 18:45:11 megatron_base_model:821] The model: MegaMolBARTModel() does not have field.name: virtual_pipeline_model_parallel_size in its cfg. Add this key to cfg or config_mapping to make to make it configurable.
[NeMo W 2024-06-19 18:45:11 megatron_base_model:821] The model: MegaMolBARTModel() does not have field.name: sequence_parallel in its cfg. Add this key to cfg or config_mapping to make to make it configurable.
[NeMo W 2024-06-19 18:45:11 megatron_base_model:821] The model: MegaMolBARTModel() does not have field.name: expert_model_parallel_size in its cfg. Add this key to cfg or config_mapping to make to make it configurable.
[NeMo W 2024-06-19 18:45:11 megatron_base_model:821] The model: MegaMolBARTModel() does not have field.name: use_cpu_initializat

[NeMo I 2024-06-19 18:45:11 tokenizer_utils:199] Using regex tokenization
[NeMo I 2024-06-19 18:45:11 regex_tokenizer:240] Loading vocabulary from file = /tmp/tmpd_7a1e8l/8bd356169fbe4824b9bf13ecdea37a8a_megamolbart.vocab
[NeMo I 2024-06-19 18:45:11 regex_tokenizer:254] Loading regex from file = /tmp/tmpd_7a1e8l/f4511a31569943ff909607eaee4f6668_megamolbart.model
[NeMo I 2024-06-19 18:45:11 megatron_base_model:315] Padded vocab_size: 640, original vocab_size: 523, dummy tokens: 117.


[NeMo W 2024-06-19 18:45:11 megatron_lm_encoder_decoder_model:240] Could not find encoder or decoder in config. This is probably because of restoring an old checkpoint. Copying shared model configs to encoder and decoder configs.
[NeMo W 2024-06-19 18:45:11 megatron_lm_encoder_decoder_model:206] bias_gelu_fusion is deprecated. Please use bias_activation_fusion instead.
[NeMo W 2024-06-19 18:45:11 megatron_lm_encoder_decoder_model:206] bias_gelu_fusion is deprecated. Please use bias_activation_fusion instead.


[NeMo I 2024-06-19 18:45:12 nlp_overrides:752] Model MegaMolBARTModel was successfully restored from /workspace/bionemo/MegaMolBART_0_2_0.nemo.
[NeMo I 2024-06-19 18:45:14 megatron_lm_encoder_decoder_model:1195] Decoding using the greedy-search method...


In [11]:
df = pd.read_excel('TestInputData_June17.xlsx')

In [12]:
df.head()

,Compound as referred to in the document,Canonical Smiles,Size,ExptVariable
0,Compound 1,CCCCCCCCCCCCCCCCCCN(CCO)CCCCCCCC(=O)OC(CCCCCCC...,72.7,402000000
1,Compound 2,CCCCCCCCCCCCCCN(CCO)CCCCCCCC(=O)OC(CCCCCCCC)CC...,83.9,4870000000
2,Compound 3,CCCCCCCCCN(CCO)CCCCCCCC(=O)OC(CCCCCCCC)CCCCCCCC,97.5,13900000000
3,Compound 4,CCCCCCCCC(CCCCCCCC)OC(=O)CCCCCCCN(CCO)CCCCCCCC,120.5,5260000000
4,Compound 5,CCCCCCCCC(CCCCCCCC)OC(=O)CCCCCCCN(CCO)CCCCCC,196.4,58400000


In [23]:
x = df['Canonical Smiles'].tolist()
y = np.log10(df['ExptVariable'])

In [30]:
split_df = {}
train_idx,test_idx = train_test_split(list(range(len(x))),test_size =0.1,shuffle  = True,random_state= 42)
train_idx,val_idx = train_test_split(train_idx,test_size =0.1,shuffle  = True,random_state= 42)
split_df['train'] = train_idx
split_df['test'] = test_idx
split_df['val'] = val_idx

In [24]:
batches = np.array_split(x,len(x)//10)

In [36]:
embeddings = []
for smis in batches:
    embeddings.append(inferer.seq_to_embeddings(smis).detach().cpu().numpy())
fp_X = np.vstack(embeddings)
fp_X.shape

(111, 512)

In [57]:
def select_random_forest(parameter_grid, X, y, split_df):
    X_train = np.array(X)[split_df["train"],]
    X_val = np.array(X)[split_df["val"],]
    X_test = np.array(X)[split_df["test"],]
    y_train = np.array(y)[split_df["train"]]
    y_val = np.array(y)[split_df["val"]]
    y_test = np.array(y)[split_df["test"]]

    val_r2 = []
    test_r2 = []
    val_mse = []
    test_mse = []
    val_mae = []
    test_mae = []
    for max_depth in parameter_grid["max_depth"]:
        for min_samples_split in parameter_grid["min_samples_split"]:
            for n_estimators in parameter_grid["n_estimators"]:
                model = RandomForestRegressor(
                    max_depth=max_depth,
                    n_estimators=n_estimators,
                    min_samples_split=min_samples_split,
                    random_state=random_seed,
                )
                model.fit(X_train, y_train)
                pred_val = model.predict(X_val)
                pred_test = model.predict(X_test)
                val_r2.append(r2_score(y_val, pred_val))
                test_r2.append(r2_score(y_test, pred_test))
                val_mse.append(mean_squared_error(y_val, pred_val))
                test_mse.append(mean_squared_error(y_test, pred_test))
                val_mae.append(mean_absolute_error(y_val, pred_val))
                test_mae.append(mean_absolute_error(y_test, pred_test))
    return (val_r2, test_r2, val_mse, test_mse, val_mae, test_mae)
def select_catboost(parameter_grid, X, y, split_df):
    X_train = np.array(X)[split_df["train"],]
    X_val = np.array(X)[split_df["val"],]
    X_test = np.array(X)[split_df["test"],]
    y_train = np.array(y)[split_df["train"]]
    y_val = np.array(y)[split_df["val"]]
    y_test = np.array(y)[split_df["test"]]

    val_r2 = []
    test_r2 = []
    val_mse = []
    test_mse = []
    val_mae = []
    test_mae = []
    for depth in parameter_grid["depth"]:
        for learning_rate in parameter_grid["learning_rate"]:
            for iterations in parameter_grid["iterations"]:
                model = CatBoostRegressor(
                    depth=depth,
                    iterations=iterations,
                    learning_rate=learning_rate,
                    verbose=False,
                    random_seed=random_seed,
                )
                model.fit(X_train, y_train)
                pred_val = model.predict(X_val)
                pred_test = model.predict(X_test)
                val_r2.append(r2_score(y_val, pred_val))
                test_r2.append(r2_score(y_test, pred_test))
                val_mse.append(mean_squared_error(y_val, pred_val))
                test_mse.append(mean_squared_error(y_test, pred_test))
                val_mae.append(mean_absolute_error(y_val, pred_val))
                test_mae.append(mean_absolute_error(y_test, pred_test))
    return (val_r2, test_r2, val_mse, test_mse, val_mae, test_mae)

In [69]:
def run_ml_model(X, y, split_df, parameter_grid,model):
    val_r2, test_r2, val_mse, test_mse, val_mae, test_mae = model(
        parameter_grid, X, y, split_df
    )
    # --- Compute the best result from the validation set ---
    best_idx = np.argmin(val_mse)
    val_r2, test_r2, val_mse, test_mse, val_mae, test_mae = (
        val_r2[best_idx],
        test_r2[best_idx],
        val_mse[best_idx],
        test_mse[best_idx],
        val_mae[best_idx],
        test_mae[best_idx],
    )
    print("VAL")
    print(f"R2 Score: {val_r2}")
    print(f"mse: {val_mse}")
    print(f"mae: {val_mae}")
    print("TEST")
    print(f"R2 Score: {test_r2}")
    print(f"mse: {test_mse}")
    print(f"mae: {test_mae}")
    # latex table row
    print(
        f"{val_r2:.3f} & {test_r2:.3f} & {val_mse:.3f} & {test_mse:.3f} & {val_mae:.3f} & {test_mae:.3f} \\"
    )

In [70]:
random_seed = 42

In [71]:
%%time
param_grid = {
    "depth": [3, 4, 5], 
    "learning_rate": [0.01, 0.1], 
    "iterations": [1000, 2000, 3000],
}
run_ml_model(fp_X, y, split_df, param_grid,select_catboost)

VAL
R2 Score: 0.7142030522522409
mse: 0.364418508474465
mae: 0.5101583107605063
TEST
R2 Score: 0.20558872716763177
mse: 1.1551897935523554
mae: 0.7935367173293248
0.714 & 0.206 & 0.364 & 1.155 & 0.510 & 0.794 \
CPU times: user 21min 2s, sys: 1min 15s, total: 22min 18s
Wall time: 3min 5s


In [72]:
%%time
param_grid = {
    "max_depth": [3, 5, 7],  # Maximum depth of the trees
    "min_samples_split": [2, 5, 10],  # Minimum samples required to split an internal node
    "n_estimators": [100, 200, 300]  # Number of trees in the forest
}
run_ml_model(fp_X, y, split_df, param_grid,select_random_forest)

VAL
R2 Score: 0.693036436316411
mse: 0.39140797309112296
mae: 0.5105700926344732
TEST
R2 Score: 0.10148948032679062
mse: 1.306565273205704
mae: 0.8265992218774452
0.693 & 0.101 & 0.391 & 1.307 & 0.511 & 0.827 \
CPU times: user 54 s, sys: 4.56 ms, total: 54 s
Wall time: 53.9 s
